# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Konstante
G = 6.67430e-11        # gravitacijska konstanta [m^3 kg^-1 s^-2]
M_Z = 5.972e24         # masa Zemlje [kg]
R_Z = 6371e3           # polmer Zemlje [m]
rho_0 = 1.225          # gostota zraka na površini [kg/m^3]
M_zraka = 28.96        # molska masa zraka [kg/kmol]
R_plin = 8314.46       # splošna plinska konstanta [J/(kmol K)]
T_0 = 288.15           # temperatura ozračja [K]

# Raketa
C_d = 0.2              # koeficient upora
S = 10.0               # prečni presek [m^2]
F_pot = 845e3          # sila potiska (Merlin 1D, morska gladina) [N]
v_izp = 2500.0         # izstopna hitrost plinov [m/s]
mdot = F_pot / v_izp   # poraba goriva [kg/s]
m_rak = 10_000         # suha masa rakete [kg]
m_tov = 1_000          # masa tovora [kg]
m_gor0 = 20_000        # začetna masa goriva [kg]

g0 = G * M_Z / R_Z**2  # težni pospešek na površini
H = 0                  # TODO: višinska skala ozračja (P1)

# Funkcije

In [ ]:
def gravitacijska_sila(h, m):
    """Gravitacijska sila na maso m na višini h nad površjem (negativna = navzdol)."""
    # TODO
    pass


def gostota(h):
    """Gostota zraka po barometrski enačbi (izotermno ozračje)."""
    # TODO
    pass


def sila_upora(h, v):
    """Sila zračnega upora, vedno nasprotna hitrosti."""
    # TODO: pazi na smer (v * |v|)
    pass


def sila_potiska(m_g):
    """Konstantna sila potiska, dokler je na voljo gorivo."""
    # TODO
    pass


def korak(st, dt, gravitacija=True, upor=True):
    """En korak eksplicitne Eulerjeve metode. st je slovar s ključi t, h, v, m_g.
    Vrne nov slovar stanja in pospešek a v začetku koraka."""
    m = m_rak + m_tov + st["m_g"]
    F = sila_potiska(st["m_g"])
    if gravitacija:
        F += gravitacijska_sila(st["h"], m)
    if upor:
        F += sila_upora(st["h"], st["v"])
    a = F / m
    # TODO: posodobi t, h, v in m_g (gorivo ne sme postati negativno)
    nov = dict(st)
    return nov, a


def simulacija(m_g0=m_gor0, dt=0.01, gravitacija=True, upor=True, konec="vrh"):
    """Simulira navpični let. konec="vrh": do najvišje točke (v < 0),
    konec="izgorevanje": do porabe goriva. Vrne DataFrame."""
    st = {"t": 0.0, "h": 0.0, "v": 0.0, "m_g": float(m_g0)}
    zapis = []
    while True:
        nov, a = korak(st, dt, gravitacija, upor)
        m = m_rak + m_tov + st["m_g"]
        zapis.append({**st, "a": a, "m": m,
                      "F_g": gravitacijska_sila(st["h"], m),
                      "F_u": sila_upora(st["h"], st["v"]),
                      "F_p": sila_potiska(st["m_g"])})
        if konec == "vrh" and nov["v"] < 0 and st["t"] > 0:
            break
        if konec == "izgorevanje" and st["m_g"] <= 0:
            break
        st = nov
    return pd.DataFrame(zapis)


def ciolkovski(m0, mk):
    """Sprememba hitrosti po raketni enačbi (brez gravitacije in upora)."""
    # TODO
    pass


def visina_vrha(h_b, v_b):
    """Najvišja točka po izgorevanju iz ohranitve energije (brez upora, g(h))."""
    # TODO: v_b^2/2 - GM/(R+h_b) = -GM/(R+h_max)
    pass

# Testno območje

In [ ]:
print(f"g0 = {g0:.3f} m/s^2, H = {H:.0f} m, mdot = {mdot:.0f} kg/s")
print("rho(0)   =", gostota(0.0))
print("rho(H)/rho0 =", gostota(H) / rho_0, "(pričakovano 1/e =", np.exp(-1), ")")
print("F_g(0, 1 kg) =", gravitacijska_sila(0.0, 1.0))
print("F_u(0, +100 m/s) =", sila_upora(0.0, 100.0), "| F_u(0, -100 m/s) =", sila_upora(0.0, -100.0))
print("čas izgorevanja t_b =", m_gor0 / mdot, "s")

# Naloga 1 - Slika 1

In [ ]:
h = np.linspace(0, 100e3, 500)
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(121)
# TODO: gostota zraka v odvisnosti od višine (os x v km)
ax = fig.add_subplot(122)
# TODO: težni pospešek g(h) = |F_g(h, 1 kg)|
fig.tight_layout()
plt.show()

rho_10km = None  # TODO: Preglednica 1
g_100km = None
print(rho_10km, g_100km)

# Naloga 2

In [ ]:
df = simulacija(m_gor0, dt=0.01)
df.head()

# Naloga 2 - Slika 2

In [ ]:
fig = plt.figure(figsize=(11, 8))
# TODO: štirje paneli: h(t), v(t), a(t) in sile F_p, F_g, F_u v odvisnosti od t
fig.tight_layout()
plt.show()

In [ ]:
# Preglednica 2
# TODO: čas izgorevanja, višina in hitrost ob izgorevanju, največji pospešek, vrh in čas do vrha
# df.to_excel("rezultati_simulacije.xlsx")  # izvoz rezultatov

# Naloga 3

In [ ]:
m0, mk = m_rak + m_tov + m_gor0, m_rak + m_tov
dv_teor = ciolkovski(m0, mk)
# TODO: tri simulacije do izgorevanja (brez g in upora, z g, z g in uporom); Preglednica 3

# Naloga 4

In [ ]:
# TODO: h_max iz ohranitve energije in primerjava s simulacijama brez upora in z njim; Preglednica 4